# Section 4: Toxic comment classification: Bi-LSTM with GloVe

Trains a bidirectional LSTM on the YouTube toxicity dataset (`istoxic` label), tunes it with Bayesian search and k-fold cross-validation, and evaluates the best model on a held-out validation set.

Pipeline: load and clean comments → tokenize and pad → GloVe embedding matrix → split → Bayesian search (each candidate scored by k-fold CV) → evaluate the best model.

## 1. Setup
Imports, the shared project code in the `functions/` package (`preprocessing`, `embeddings`, `cv`, `tuning`), and a filter for a harmless Keras warning about masking.

In [ ]:
# Add the parent directory ('Project Template') to Python's search path
import sys
sys.path.append('..')

# project code: see functions/__init__.py for what each module contains
from functions.preprocessing import cleanup_text
from functions.embeddings import build_embedding_matrix, download_glove, glove_filename
from functions.cv import kfold_wrapper
from functions.tuning import bayes_search


import os
import re
import string
import warnings
import numpy as np
import unicodedata
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Embedding
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
from skopt.space import Categorical, Integer, Real
import matplotlib.pyplot as plt

# the nightly Keras build wrongly warns that LSTM / Bidirectional destroy the mask (masking works)
warnings.filterwarnings('ignore', message='.*was passed an input with a mask attached.*')

## 2. Load data
Reads the labelled comments and normalises the column names (lowercase, underscores) so they can be referenced consistently.

In [ ]:
## load data
df = pd.read_csv('../dataset/youtoxic_english_1000.csv')

## clean up column names
df.columns = [i.strip().replace(' ', '_').lower() for i in df.columns]
df.head(5)

## 3. Clean the comments
Chooses the text and label columns, lowercases the comments and applies `cleanup_text` (HTML and URL removal, contraction expansion, punctuation handling) so the words match what GloVe knows. The cleaned text overwrites the original column, so everything below works on cleaned comments.

In [ ]:
commentsCol = 'text'
labelCol = 'istoxic'


## clean up comments for tokenization and vector embedding
df[commentsCol] = df[commentsCol].str.lower()
df[commentsCol] = df[commentsCol].apply(lambda x: cleanup_text(x))


## 4. Explore the data
Shows the class balance of `istoxic` and the distribution of comment lengths (in words), with min, max and mean marked. The length distribution informs the padding length (`max_length`) chosen in the next step.

In [ ]:
counts = df[labelCol].value_counts()
total = len(df)

fig, ax = plt.subplots(figsize=(10, 5), nrows = 1,ncols = 2)

# plot distribution of toxic comments
bars = ax[0].bar(counts.index.astype(str), counts.values, color=['#4C72B0', '#DD8452'])
ax[0].set_xlabel(labelCol)
ax[0].set_ylabel('Count')
ax[0].set_title(f'Distribution of `{labelCol}`')

for bar, val in zip(bars, counts.values):
    ax[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
               f'{val}\n({val/total:.1%})', ha='center', va='bottom', fontsize=10)


# plot distribution of comment lengths
ax[1].hist(df[commentsCol].str.split().apply(len),bins = 10, color='#4C72B0', edgecolor='black')

ax[1].set_xlabel('Comment Length')
ax[1].set_ylabel('Count')
ax[1].set_title(f'Distribution of `{commentsCol}` Lengths')

# draw min, max, and mean lines
ax[1].axvline(df[commentsCol].str.split().apply(len).min(), color='red', linestyle='dashed', linewidth=1, label='Min')
ax[1].axvline(df[commentsCol].str.split().apply(len).max(), color='blue', linestyle='dashed', linewidth=1, label='Max')
ax[1].axvline(df[commentsCol].str.split().apply(len).mean(), color='green', linestyle='dashed', linewidth=1, label='Mean')
ax[1].legend()


# label min max and mean
ax[1].text(df[commentsCol].str.split().apply(len).min(), ax[1].get_ylim()[1]*0.9, f'Min: {df[commentsCol].str.split().apply(len).min()}', color='red', fontsize=10)
ax[1].text(df[commentsCol].str.split().apply(len).max(), ax[1].get_ylim()[1]*0.9, f'Max: {df[commentsCol].str.split().apply(len).max()}', color='blue', fontsize=10)
ax[1].text(df[commentsCol].str.split().apply(len).mean(), ax[1].get_ylim()[1]*0.9, f'Mean: {df[commentsCol].str.split().apply(len).mean():.2f}', color='green', fontsize=10)
plt.tight_layout()
plt.show()

## 5. Tokenize and pad
Sets the vocabulary size, the padded length and the embedding dimension, which are reused by the model and the inference function. The tokenizer maps each word to an integer id. Sequences are padded or truncated at the end to a fixed `max_length`, with 0 reserved for padding.

`embed_dim` must match the GloVe file loaded next.

In [ ]:
# Initialize the tokenizer
vocab_size = 10000
max_length = 80
embed_dim = 100  # must match the GloVe file (50, 100, 200 or 300)

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')

# Fit the tokenizer on your cleaned texts (this learns the vocabulary)
tokenizer.fit_on_texts(df[commentsCol])

# Convert texts to sequences of integers
sequences = tokenizer.texts_to_sequences(df[commentsCol])

# Example of what this looks like
print('Original:', df[commentsCol].iloc[0])
print('\nTokenized sequence:', sequences[0])

# Pad sequences
padded_sequences = pad_sequences(
    sequences, maxlen=max_length, padding='post', truncating='post'
)

print('\nPadded shape:', padded_sequences.shape)

## 6. Load GloVe embeddings
Builds the embedding matrix that initialises the model's embedding layer: row *i* holds the pre-trained GloVe vector of the word with id *i*. It uses the local copy in `../glove` and downloads the file there first if it is missing. Words GloVe doesn't know get small random vectors. The printed coverage is the share of distinct vocabulary words found in GloVe.

In [ ]:
# Load pre-trained GloVe vectors, preferring the local copy in ../glove
glove_dir = '../glove'
glove_path = os.path.join(glove_dir, glove_filename(embed_dim))

try:
    embedding_matrix, glove_coverage = build_embedding_matrix(glove_path, tokenizer.word_index, vocab_size)
except FileNotFoundError:
    print(f'{glove_path} not found.')
    download_glove(embed_dim, glove_dir)
    embedding_matrix, glove_coverage = build_embedding_matrix(glove_path, tokenizer.word_index, vocab_size)

print('Embedding matrix shape:', embedding_matrix.shape)
print(f'Vocabulary found in GloVe: {glove_coverage:.1%}')

## 7. Reference model
A fixed example of the architecture: GloVe-initialised embedding (frozen, with padding masked) → Bi-LSTM → sigmoid output, compiled with several metrics. The search below rebuilds the model itself through `build_model`, so this cell is not needed for the search or for evaluating the best model. It shows the architecture and its summary.

In [ ]:
# 1. Define parameters based on tokenization step
v_size = (
    vocab_size  # Max number of unique words your tokenizer tracked (e.g., top 5,000)
)
embedding_dim = (
    embed_dim  # The number of dimensions (coordinates) for each word vector
)
m_length = (
    max_length  # The fixed length of your padded sequences (e.g., 50 words per comment)
)

# 2. Build the Model Architecture
model = Sequential([
    # The Embedding Layer: integer-to-vector lookup, initialised with GloVe and frozen;
    # mask_zero makes the LSTM skip the padding (index 0)
    Embedding(
        input_dim=v_size,
        output_dim=embedding_dim,
        embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
        trainable=False,
        mask_zero=True,
    ),
    # The Bi-LSTM Layer: reads the sequence forwards and backwards, then concatenates both outputs
    Bidirectional(LSTM(8, dropout=0.2, recurrent_dropout=0.2)),
    # The Output Layer: Binary classification (toxic = 1, not toxic = 0)
    Dense(1, activation='sigmoid'),
])

# 3. Compile the model
model.compile(
loss='binary_crossentropy',
    optimizer='adam',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),  # Recall = True Positive Rate
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.F1Score(name='f1_score', threshold=0.5)
    ]
)

# Print the model summary to see the shapes match up
model.summary()

## 8. Train / validation split
Holds out 20% of the comments as a validation set that is never used for training or tuning. K-fold cross-validation then runs inside the 80% training part only. The commented-out block is a quick manual test of `kfold_wrapper` on the reference model.

In [ ]:
# split data into training and validation
X_train, X_val, y_train, y_val = train_test_split(
    padded_sequences, df[labelCol], test_size=0.2, random_state=42
)

# test the k-fold cross-validation wrapper function
'''
print("Evaluating model...")
evaluated_object = kfold_wrapper(model, X_train, y_train, X_val, y_val, n_splits = 5, epochs = 5, batch_size = 16)
print(evaluated_object.avg_train_metrics)
print(evaluated_object.validation_metrics)
'''

## 9. Hyperparameter search
Defines how a model is built from a set of hyperparameters (`build_model`) and the search space (embedding trainable or frozen, LSTM size, dropout, learning rate). The Bayesian search scores each candidate with stratified k-fold CV on the training set and maximises the average validation PR-AUC.

**Choosing the model.** The top trials are usually within noise of each other, so the search selects, among the trials whose mean PR-AUC is within one fold standard deviation of the best, the one with the lowest standard deviation across folds (`selection='stable'`). It returns that trial's `kfold_wrapper`, whose final model is trained on the full training set and scored on the validation set.

**Choosing the threshold.** The model's cutoff is chosen on the out-of-fold predictions of the training set: the highest-recall cutoff whose false positive rate (the share of non-toxic comments flagged) stays at or below `MAX_FPR`. The cap reflects how many false alarms moderators can review. A recall-weighted score such as F2 was tried first, but it ran to the lowest cutoff and flagged most comments. The tuned value becomes the model's `threshold`, so the confusion matrix, ROC marker, error table and `toxicity_check` below all use it. The tuning needs no re-training, so it also runs on a loaded result and you can change `MAX_FPR` without re-running the search. The hold-out numbers at the default 0.5 cutoff are printed too, for comparison.

If a saved result exists in `../models/bayes_search`, you are asked whether to load it or re-run the search. Choose to re-run after changing the data, preprocessing or search space, because a loaded result is not checked against them.

In [ ]:
def build_model(trainable, lstm_units, dropout, lr):
    m = Sequential([
        Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim,
            embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
            trainable=trainable,
            mask_zero=True,  # LSTM skips the padding (index 0)
        ),
        Bidirectional(LSTM(lstm_units, dropout=dropout)),
        Dense(1, activation='sigmoid'),
    ])
    m.compile(
        loss='binary_crossentropy',
        optimizer=tf.keras.optimizers.Adam(lr),
        metrics=[
            'accuracy',
            tf.keras.metrics.F1Score(name='f1_score', threshold=0.5),
            tf.keras.metrics.AUC(curve='PR', name='pr_auc'),
        ],
    )
    return m

search_space = {
    'trainable': Categorical([True, False]),  # fine-tune GloVe or keep it frozen
    'lstm_units': Integer(8, 128, prior='log-uniform'),
    'dropout': Real(0.0, 0.5),
    'lr': Real(3e-4, 1e-2, prior='log-uniform'),
    # 'epochs': Integer(10, 60),  # enable after the 5-epoch baseline, and drop epochs from kfold_kwargs
}

# Decision threshold: the highest-recall cutoff whose false positive rate (share of non-toxic
# comments flagged) stays at or below this cap, measured on out-of-fold predictions.
# Tuning needs no re-training, so changing the cap does not require re-running the search.
MAX_FPR = 0.35

# a previous run is saved here; the user chooses whether to load it or re-run the search
save_dir = '../models/bayes_search'
rerun = True
if os.path.exists(os.path.join(save_dir, 'search_state.pkl')):
    answer = input('Found a previously optimised model. Load it [Y] or re-run the optimisation [N]? (default: Y) ').strip().lower()
    rerun = answer.lower().startswith('n')

if rerun:
    search = bayes_search(
        build_model, search_space, X_train, y_train, X_val, y_val,
        n_iter=30, n_initial_points=10, objective='pr_auc',
        kfold_kwargs={'n_splits': 5, 'epochs': 5, 'batch_size': 16},
        selection='stable',  # among near-ties on mean PR-AUC, take the lowest std across folds
    )
    best_kf = search.fit()
else:
    search = bayes_search.load(save_dir)
    best_kf = search.best_kfold
    print(f'Loaded from {save_dir}')

# the validation set is not used to choose the threshold
if getattr(best_kf, 'oof_proba', None) is not None:
    tuning = best_kf.tune_threshold(max_fpr=MAX_FPR)
else:
    tuning = None
    print('No out-of-fold predictions in this saved result: re-run the search to tune the threshold.')

if rerun:
    search.save(save_dir)
    print(f'Saved to {save_dir}')

print('Selected trial:', getattr(search, 'best_trial', None), '| params:', search.best_params)
print('Hold-out @ threshold 0.5:  ', {k: round(v, 4) for k, v in best_kf.validation_metrics.items()})
if tuning is not None:
    print(f"Tuned threshold: {tuning['threshold']:.3f} ({tuning['rule']}); "
          f"out-of-fold recall {tuning['oof_recall']:.2f}, precision {tuning['oof_precision']:.2f}, FPR {tuning['oof_fpr']:.2f}")
    print('Hold-out @ tuned threshold:', {k: round(v, 4) for k, v in best_kf.tuned_validation_metrics.items()})
search.results.sort_values('avg_val_pr_auc', ascending=False).head()

## 10. Search progress
Validation PR-AUC (mean ± standard deviation across the CV folds) for every round of the search. The dotted line marks the end of the random initial points. Use it to see whether the search is still improving and how large the fold-to-fold noise is relative to the differences between rounds.

In [ ]:
# mean +/- std (across the CV folds) of validation PR-AUC for each round of the search
res = search.results
mean, std = res['avg_val_pr_auc'], res['std_val_pr_auc']

fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar(res.index, mean, yerr=std, fmt='o-', color='#4C72B0', ecolor='#9db4d6',
            capsize=3, label='mean ± std across folds')
ax.axvline(search.n_initial_points + 0.5, color='grey', linestyle=':', label='end of random initial points')
selected_trial = getattr(search, 'best_trial', None) or mean.idxmax()
ax.scatter([selected_trial], [mean[selected_trial]], color='#DD8452', zorder=3, s=80, label=f'selected (trial {selected_trial})')
ax.set_xlabel('Search round')
ax.set_ylabel('Validation PR-AUC')
ax.set_title('PR-AUC per round of Bayesian search')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Evaluate the best model: confusion matrix
Predicts the held-out validation set with the best model and shows the confusion matrix at the model's threshold. It shows how many toxic comments are caught and how many clean ones are flagged.

In [ ]:
# confusion matrix of the best model on the validation set
from sklearn.metrics import ConfusionMatrixDisplay

proba = best_kf.final_model.predict(X_val, verbose=0).ravel()
pred = (proba >= best_kf.threshold).astype(int)

ConfusionMatrixDisplay.from_predictions(
    np.asarray(y_val).astype(int),
    pred,
    display_labels=['Non-toxic', 'Toxic'],
    cmap='Blues',
)
plt.title('Best model: validation confusion matrix')
plt.show()

### ROC curve and AUC
Plots the true positive rate (share of toxic comments caught) against the false positive rate (share of clean comments flagged) as the cutoff moves from 1 to 0. The AUC summarises how well the model ranks toxic above clean comments, independent of any threshold; 0.5 is chance and 1.0 is perfect. The orange dot marks where the model's current threshold sits on the curve, so moving the threshold slides that point along the curve.

In [ ]:
# ROC curve of the best model on the validation set
from sklearn.metrics import roc_curve, roc_auc_score

proba = best_kf.final_model.predict(X_val, verbose=0).ravel()
actual = np.asarray(y_val).astype(int)

fpr, tpr, _ = roc_curve(actual, proba)
auc = roc_auc_score(actual, proba)

# operating point at the model's threshold
pred = (proba >= best_kf.threshold).astype(int)
op_tpr = ((pred == 1) & (actual == 1)).sum() / (actual == 1).sum()
op_fpr = ((pred == 1) & (actual == 0)).sum() / (actual == 0).sum()

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, color='#4C72B0', linewidth=2, label=f'Bi-LSTM (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], color='grey', linestyle='--', label='Chance (AUC = 0.5)')
ax.scatter([op_fpr], [op_tpr], color='#DD8452', zorder=3, s=70, label=f'Threshold = {best_kf.threshold:g}')
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate (recall)')
ax.set_title('Best model: validation ROC curve')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Cutoff trade-off (out-of-fold)
Shows what each cutoff does to recall, precision, false positive rate and the share of comments flagged, on the out-of-fold predictions of the training set. The row marked `tuned` is the cutoff chosen by the false positive cap. Lowering the cutoff catches more toxic comments but flags more clean ones, so use this table to judge whether the extra recall is worth the extra moderation work. The validation columns are a sanity check only. Validation has a higher share of toxic comments than training, so its precision looks better, and on real r/jobs traffic, where toxic comments are likely rarer, precision would be lower still.

In [ ]:
# Trade-off between catching toxic comments and false alarms at different cutoffs.
# oof_* : out-of-fold predictions on the training set, the basis for choosing the cutoff
# val_* : validation set, for checking only (it did not influence the cutoff)
# recall = share of toxic comments caught | precision = share of flagged comments that are toxic
# fpr = share of non-toxic comments flagged | flagged = share of all comments flagged
best_kf.threshold_table().round(2)

## 12. Error analysis
Lists the mis-classified validation comments with the true label, predicted label, predicted probability, cleaned text and length in tokens, sorted by probability. Use it to look for patterns in the errors, such as truncated long comments, sarcasm or implicit toxicity, and noisy labels.

In [ ]:
# mis-classified validation comments of the best model
proba = best_kf.final_model.predict(X_val, verbose=0).ravel()
pred = (proba >= best_kf.threshold).astype(int)
actual = np.asarray(y_val).astype(int)

errors = pd.DataFrame({
    'actual_label': actual,
    'predicted_label': pred,
    'predicted_probability': proba,
    'comment': df.loc[y_val.index, commentsCol].values,  # cleaned text, as the model saw it
    'n_tokens': df.loc[y_val.index, commentsCol].str.split().str.len().values,
}, index=y_val.index)

errors = errors[errors['actual_label'] != errors['predicted_label']]
errors['error_type'] = np.where(errors['actual_label'] == 1, 'missed toxic (FN)', 'false alarm (FP)')

print(errors['error_type'].value_counts().to_string())
pd.set_option('display.max_colwidth', 200)
errors.sort_values('predicted_probability')

## 13. Predict on new comments
`toxicity_check` applies the same pipeline to new text (clean → tokenize → pad → predict) with the best model and returns "Toxic" or "Non-toxic" using the model's threshold, or one you pass in. It depends on the tokenizer, `max_length` and `cleanup_text` above, so those must match what the model was trained with.

In [ ]:
## pipeline to predict new data
def toxicity_check(new_comments, best_kf, threshold=None):
    # Use the wrapper's cutoff unless another one is given (e.g. a tuned threshold)
    threshold = best_kf.threshold if threshold is None else threshold

    # Clean the new comments
    cleaned_comments = [cleanup_text(comment.lower()) for comment in new_comments]

    # Convert to sequences
    sequences = tokenizer.texts_to_sequences(cleaned_comments)

    # Pad sequences
    padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

    # Predict with the trained model from the search
    probabilities = best_kf.final_model.predict(padded_sequences, verbose=0).ravel()

    return [{'label': 'Toxic' if p.round(4) >= threshold else 'Non-toxic', 'p': p} for p in probabilities]

### Testing
Runs `toxicity_check` on a sample comment as a quick sanity check.

In [ ]:
toxicity_check(["wouldnt it be nice if you got hit by a bus?",
                "im so sorry to hear that you got hit by a bus",
                "I don’t mean to be insensitive with this question but isn’t it hard to be fired from a government job?",
                "you've got to be a genius to be fired from a government job",
                "you're as smart as a rock"], best_kf)